# K-means clustering

Group students by selected behavior and prior-grade features, then inspect cluster profiles.

**Dataset:** `data/student_data.csv` (395 student records, 33 columns). **Target:** final grade `G3` (0–20). For classification labs, pass means `G3 ≥ 10`. Earlier grades `G1` and `G2` are included as predictors; interpret results as prediction after those earlier grades are known. The split is fixed with random seed 42 for reproducibility.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATHS = [Path("data/student_data.csv"), Path("../data/student_data.csv")]
DATA_PATH = next((p for p in DATA_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place student_data.csv in the repository's data/ folder.")
df = pd.read_csv(DATA_PATH)
TARGET = "G3"  # final-period grade (0–20)
PASS_THRESHOLD = 10
numeric_features = df.select_dtypes(include=np.number).columns.drop(TARGET).tolist()
categorical_features = df.select_dtypes(exclude=np.number).columns.tolist()
feature_cols = numeric_features + categorical_features
X = df[feature_cols]
y_reg = df[TARGET]
y_class = (df[TARGET] >= PASS_THRESHOLD).astype(int)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_features=["age","studytime","failures","absences","G1","G2"]
X_cluster=df[cluster_features].copy()
X_scaled=StandardScaler().fit_transform(X_cluster)
rows=[]
for k in range(2,7):
    labels=KMeans(n_clusters=k,n_init=20,random_state=42).fit_predict(X_scaled)
    rows.append({"k":k,"inertia":KMeans(n_clusters=k,n_init=20,random_state=42).fit(X_scaled).inertia_,"silhouette":silhouette_score(X_scaled,labels)})
metrics=pd.DataFrame(rows); display(metrics)
plt.plot(metrics.k,metrics.inertia,marker="o"); plt.xlabel("k"); plt.ylabel("Inertia"); plt.title("K-means elbow curve"); plt.show()
best_k=int(metrics.loc[metrics.silhouette.idxmax(),"k"])
df["cluster"]=KMeans(n_clusters=best_k,n_init=20,random_state=42).fit_predict(X_scaled)
display(df.groupby("cluster")[cluster_features+["G3"]].mean().round(2))
sns.scatterplot(data=df,x="G1",y="G2",hue="cluster",palette="tab10"); plt.title(f"Student clusters (k={best_k})"); plt.show()
